# FAIR Data Science model notebook

This notebook is the runnable project implementation for the football-player market-value experiment.

It reads the two input Excel files from the repository `data/` folder, trains one XGBoost regression model, and writes the model and result files used by the Model Card and RO-Crate.


In [17]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

RANDOM_STATE = 42

ROOT = Path.cwd()
DATA_DIR = ROOT.cwd().parent / "data"

MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)


In [18]:
transfer_path = DATA_DIR / "Nisanov_final_data.xlsx"
soccerplayers_path = DATA_DIR / "Dataset soccerplayers.xlsx"

required_files = [transfer_path, soccerplayers_path]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s): " + ", ".join(missing_files) +
        ". Put the full, non-short Excel files into the repository data/ folder."
    )

print("Transfer dataset:", transfer_path)
print("Soccerplayers dataset:", soccerplayers_path)


Transfer dataset: c:\Users\Konrad\Documents\Sonstiges\Master\SS2026\Data Stewardship UE\fair-ds-experiment\data\Nisanov_final_data.xlsx
Soccerplayers dataset: c:\Users\Konrad\Documents\Sonstiges\Master\SS2026\Data Stewardship UE\fair-ds-experiment\data\Dataset soccerplayers.xlsx


In [19]:
%pip install openpyxl

# Load all yearly sheets from the Transfer Value Determinants workbook.
transfer_sheets = pd.read_excel(transfer_path, sheet_name=None)

transfer_data = (
    pd.concat(
        [sheet.dropna(axis=1, how="all") for sheet in transfer_sheets.values()],
        ignore_index=True,
        sort=False,
    )
)

# Load the second source dataset as reused input data.
# It is not needed for this minimal training run, but it is part of the documented project inputs.
soccerplayers_data = pd.read_excel(soccerplayers_path)

print("Transfer data shape:", transfer_data.shape)
print("Soccerplayers data shape:", soccerplayers_data.shape)
transfer_data.head()



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Transfer data shape: (2502, 22)
Soccerplayers data shape: (438, 12)


,player_name,position,nationality,age_then,age_now,club_name,club_performance,relegation,success_or_not,total_games,...,total_minutes,total_goals,height,start_value,end_value,delta_value,value_0_mln,value_end_mln,value_delta_mln,season
0,Jefferson Lerma,Defensive Midfield,Colombia,25,30,AFC Bournemouth,-1.0,1.0,-1.0,33,...,2704,1,179,25000000.0,20000000.0,-5000000,25.0,20.0,-5.0,2019
1,Philip Billing,Central Midfield,Denmark,24,28,AFC Bournemouth,-1.0,1.0,-1.0,34,...,2530,1,193,15000000.0,14500000.0,-500000,15.0,14.5,-0.5,2019
2,Junior Stanislas,Left Winger,England,30,35,AFC Bournemouth,-1.0,1.0,-1.0,15,...,712,3,183,3500000.0,2000000.0,-1500000,3.5,2.0,-1.5,2019
3,Dan Gosling,Central Midfield,England,30,34,AFC Bournemouth,-1.0,1.0,-1.0,24,...,1264,3,180,2500000.0,2000000.0,-500000,2.5,2.0,-0.5,2019
4,Dominic Solanke,Centre-Forward,England,22,27,AFC Bournemouth,-1.0,1.0,-1.0,32,...,1651,3,186,15000000.0,12000000.0,-3000000,15.0,12.0,-3.0,2019


## Model training

The model predicts `value_end_mln`, i.e. the final market value in million euros. The notebook uses one XGBoost regression model because this matches the Model Card.


In [20]:
target = "value_end_mln"

feature_columns = [
    "position",
    "nationality",
    "club_name",
    "age_then",
    "age_now",
    "total_games",
    "assists",
    "penalty_kicks",
    "total_minutes",
    "total_goals",
    "height",
    "start_value",
    "value_0_mln",
    "season",
]

model_data = transfer_data[feature_columns + [target, "player_name"]].copy()
model_data = model_data.dropna(subset=[target])

X = model_data[feature_columns]
y = model_data[target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=20,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=1,
    tree_method="hist",
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model),
])

X_train, X_test, y_train, y_test, train_index, test_index = train_test_split(
    X,
    y,
    model_data.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)


In [21]:
mse = mean_squared_error(y_test, predictions)
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test, predictions))
r2 = float(r2_score(y_test, predictions))

metrics = pd.DataFrame([
    {
        "model": "XGBRegressor",
        "target": target,
        "r2": r2,
        "mse": float(mse),
        "rmse": rmse,
        "mae": mae,
        "test_size": len(y_test),
        "train_size": len(y_train),
    }
])

metrics


,model,target,r2,mse,rmse,mae,test_size,train_size
0,XGBRegressor,value_end_mln,0.870226,65.539616,8.095654,5.563473,751,1751


In [22]:
prediction_output = model_data.loc[test_index, ["player_name"]].copy()
prediction_output["actual_value_end_mln"] = y_test.to_numpy()
prediction_output["predicted_value_end_mln"] = predictions
prediction_output["residual_value_end_mln"] = prediction_output["actual_value_end_mln"] - prediction_output["predicted_value_end_mln"]

model_path = MODEL_DIR / "final_model.pkl"
metrics_path = OUTPUT_DIR / "evaluation_metrics.csv"
predictions_path = OUTPUT_DIR / "predictions.csv"

joblib.dump(pipeline, model_path)
metrics.to_csv(metrics_path, index=False)
prediction_output.to_csv(predictions_path, index=False)

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)
print("Saved predictions:", predictions_path)


Saved model: c:\Users\Konrad\Documents\Sonstiges\Master\SS2026\Data Stewardship UE\fair-ds-experiment\notebooks\models\final_model.pkl
Saved metrics: c:\Users\Konrad\Documents\Sonstiges\Master\SS2026\Data Stewardship UE\fair-ds-experiment\notebooks\outputs\evaluation_metrics.csv
Saved predictions: c:\Users\Konrad\Documents\Sonstiges\Master\SS2026\Data Stewardship UE\fair-ds-experiment\notebooks\outputs\predictions.csv


## Files produced

This notebook produces the files referenced by the Model Card and RO-Crate:

- `models/final_model.pkl`
- `outputs/evaluation_metrics.csv`
- `outputs/predictions.csv`
